 Part 3: Backtesting & Performance Analysis

---

## 📋 Project Overview

**Objective:** Simulate our trading strategy and evaluate its performance with realistic constraints.

---

### Part 3 Goals:
1. **Backtesting Framework** - Build a realistic trading simulator
2. **Performance Metrics** - Calculate Sharpe, Drawdown, Turnover, etc.
3. **Transaction Cost Analysis** - Impact of trading costs
4. **Benchmark Comparison** - Compare vs Equal-Weight portfolio
5. **Visualization** - Create comprehensive performance charts

---

### Simulation Constraints (from task):
- **Initial Capital:** $1,000,000
- **Transaction Costs:** 0.10% (10 bps) per trade
- **Universe:** N stocks from the dataset

---

## 📦 Step 1: Import Libraries

In [ ]:
# Install required packages (uncomment for Colab)
# !pip install pandas numpy matplotlib seaborn scipy --quiet

# Core Libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

# Statistical analysis
from scipy import stats
from scipy.stats import skew, kurtosis

print("✅ Libraries imported successfully!")
print(f"📅 Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 📥 Step 2: Load Data and Signals

In [ ]:
# Load the trading signals from Part 2
signals = None
prices = None

try:
    signals = pd.read_csv('trading_signals_test.csv')
    print(f"✅ Test signals loaded: {len(signals):,} rows")
except FileNotFoundError:
    print("❌ Signals file not found. Please run Part 2 first.")
    raise FileNotFoundError("trading_signals_test.csv not found. Run Part 2 to generate signals.")

# Load the original price data
try:
    prices = pd.read_csv('daily_prices.csv')
    print(f"✅ Price data loaded: {len(prices):,} rows")
except FileNotFoundError:
    try:
        prices = pd.read_csv('processed_features.csv')
        print(f"✅ Processed data loaded: {len(prices):,} rows")
    except FileNotFoundError:
        print("❌ Price data not found. Please upload the data file.")
        raise FileNotFoundError("No price data found. Upload daily_prices.csv or processed_features.csv")

In [ ]:
# Identify column names with flexible pattern matching
import re

ticker_col = None
date_col = None
close_col = None

# Flexible patterns for column detection
ticker_patterns = ['ticker', 'symbol', 'stock', 'asset', 'name', 'security']
date_patterns = ['date', 'datetime', 'time', 'timestamp', 'trading_date']
close_patterns = ['close', 'price', 'adj', 'adjusted', 'last']

for col in prices.columns:
    col_lower = col.lower()
    
    # Check for ticker column
    if ticker_col is None:
        for pattern in ticker_patterns:
            if pattern in col_lower:
                ticker_col = col
                break
    
    # Check for date column  
    if date_col is None:
        for pattern in date_patterns:
            if pattern in col_lower:
                date_col = col
                break
    
    # Check for close/price column
    if close_col is None:
        for pattern in close_patterns:
            if pattern in col_lower:
                close_col = col
                break

# If still not found, try signals DataFrame for column names
if ticker_col is None or date_col is None:
    for col in signals.columns:
        col_lower = col.lower()
        if ticker_col is None:
            for pattern in ticker_patterns:
                if pattern in col_lower:
                    ticker_col = col
                    break
        if date_col is None:
            for pattern in date_patterns:
                if pattern in col_lower:
                    date_col = col
                    break

print(f"📌 Column mapping:")
print(f"   Ticker: {ticker_col}")
print(f"   Date: {date_col}")
print(f"   Close: {close_col}")
print(f"   Available columns in prices: {list(prices.columns)}")
print(f"   Available columns in signals: {list(signals.columns)}")

# Validate columns exist
if ticker_col is None:
    raise ValueError(f"Could not find ticker column. Available: {list(prices.columns)}")
if date_col is None:
    raise ValueError(f"Could not find date column. Available: {list(prices.columns)}")
if close_col is None:
    raise ValueError(f"Could not find close/price column. Available: {list(prices.columns)}")

# Convert dates
prices[date_col] = pd.to_datetime(prices[date_col])
signals[date_col] = pd.to_datetime(signals[date_col])

---

## 🏗️ Step 3: Build Backtesting Framework

Our backtester will simulate realistic trading:
- Execute trades at next day's close (to avoid look-ahead bias)
- Deduct transaction costs on each trade
- Track portfolio value, positions, and PnL daily

In [ ]:
class Backtester:
    """
    A comprehensive backtesting engine for portfolio strategies.
    
    Features:
    - Long-only or Long-short portfolio support
    - Transaction cost modeling
    - Detailed performance metrics
    - Position tracking
    - WEEKLY REBALANCING to reduce turnover (key improvement!)
    """
    
    def __init__(self, initial_capital=1_000_000, transaction_cost=0.001, rebalance_frequency=5):
        """
        Initialize the backtester.
        
        Parameters:
        -----------
        initial_capital : float
            Starting capital in dollars
        transaction_cost : float
            Cost per trade as a fraction (0.001 = 10 bps)
        rebalance_frequency : int
            Only rebalance every N days (5 = weekly, reduces turnover!)
        """
        self.initial_capital = initial_capital
        self.transaction_cost = transaction_cost
        self.rebalance_frequency = rebalance_frequency
        
        # Results storage
        self.portfolio_values = []
        self.daily_returns = []
        self.positions = {}
        self.trades = []
        self.daily_pnl = []
        self.transaction_costs_paid = []
        
    def run(self, signals_df, prices_df, ticker_col, date_col, close_col):
        """
        Run the backtest with WEEKLY REBALANCING to reduce turnover.
        
        Parameters:
        -----------
        signals_df : DataFrame
            Trading signals with columns: ticker, date, position
        prices_df : DataFrame
            Price data with columns: ticker, date, close
        ticker_col, date_col, close_col : str
            Column names
        
        Returns:
        --------
        dict with performance metrics
        """
        
        print("🚀 Starting Backtest...")
        print("="*60)
        print(f"📊 Rebalancing every {self.rebalance_frequency} days (reduces turnover!)")
        
        # Ensure dates are datetime
        signals_df = signals_df.copy()
        prices_df = prices_df.copy()
        signals_df[date_col] = pd.to_datetime(signals_df[date_col]).dt.normalize()
        prices_df[date_col] = pd.to_datetime(prices_df[date_col]).dt.normalize()
        
        # Prepare price data
        price_pivot = prices_df.pivot_table(index=date_col, columns=ticker_col, values=close_col, aggfunc='first')
        price_pivot = price_pivot.sort_index()
        
        # Get daily returns for each stock
        returns_pivot = price_pivot.pct_change()
        
        # Get unique dates in signals that also exist in prices
        signal_dates = sorted(signals_df[date_col].unique())
        valid_dates = [d for d in signal_dates if d in price_pivot.index]
        
        print(f"📅 Signal dates: {len(signal_dates)}, Valid dates with prices: {len(valid_dates)}")
        
        if len(valid_dates) == 0:
            print("⚠️ No matching dates between signals and prices!")
            self.results_df = pd.DataFrame()
            return {}
            
        # Initialize
        portfolio_value = self.initial_capital
        current_weights = {}  # ticker: weight (HOLD these between rebalances)
        
        results = []
        total_costs = 0
        total_turnover = 0
        last_rebalance_day = -self.rebalance_frequency  # Force rebalance on first day
        
        print(f"📅 Trading period: {valid_dates[0]} to {valid_dates[-1]}")
        print(f"📊 Total trading days: {len(valid_dates)}")
        
        for i, date in enumerate(valid_dates):
            # Get today's returns (for calculating portfolio performance)
            if date in returns_pivot.index and i > 0:
                today_returns = returns_pivot.loc[date]
                
                # Calculate portfolio return from CURRENT positions (held from last rebalance)
                portfolio_return = 0
                total_weight = 0
                for ticker, weight in current_weights.items():
                    if ticker in today_returns.index and not pd.isna(today_returns[ticker]):
                        portfolio_return += weight * today_returns[ticker]
                        total_weight += abs(weight)
                
                # Apply daily return to portfolio
                if total_weight > 0:
                    portfolio_value = portfolio_value * (1 + portfolio_return)
            
            # Check if it's time to rebalance (every N days)
            days_since_rebalance = i - last_rebalance_day
            should_rebalance = (days_since_rebalance >= self.rebalance_frequency) or (i == 0)
            
            daily_costs = 0
            daily_turnover_pct = 0
            n_long = len([w for w in current_weights.values() if w > 0])
            
            if should_rebalance:
                last_rebalance_day = i
                
                # Get today's signals
                daily_signals = signals_df[signals_df[date_col] == date]
                
                # Determine target positions based on signals
                long_stocks = daily_signals[daily_signals['position'] == 1][ticker_col].tolist()
                
                # Filter to stocks with valid prices
                valid_longs = [t for t in long_stocks if t in price_pivot.columns 
                              and date in price_pivot.index
                              and not pd.isna(price_pivot.loc[date, t])]
                
                n_long = len(valid_longs)
                
                # Equal weight allocation
                target_weights = {}
                if n_long > 0:
                    weight_per_stock = 1.0 / n_long
                    for ticker in valid_longs:
                        target_weights[ticker] = weight_per_stock
                
                # Calculate turnover (only happens on rebalance days!)
                all_tickers = set(current_weights.keys()) | set(target_weights.keys())
                
                for ticker in all_tickers:
                    old_weight = current_weights.get(ticker, 0)
                    new_weight = target_weights.get(ticker, 0)
                    weight_change = abs(new_weight - old_weight)
                    daily_turnover_pct += weight_change
                
                # Calculate transaction costs
                daily_turnover_dollars = daily_turnover_pct * portfolio_value
                daily_costs = daily_turnover_dollars * self.transaction_cost
                
                # Deduct transaction costs
                portfolio_value -= daily_costs
                portfolio_value = max(portfolio_value, 1)
                
                total_costs += daily_costs
                total_turnover += daily_turnover_dollars
                
                # Update weights (HOLD these until next rebalance)
                current_weights = target_weights.copy()
            
            # Store results
            results.append({
                'date': date,
                'portfolio_value': portfolio_value,
                'n_long': n_long,
                'n_short': 0,
                'transaction_costs': daily_costs,
                'turnover': daily_turnover_pct * portfolio_value,
                'rebalanced': should_rebalance
            })
        
        if len(results) == 0:
            print("❌ No valid trading days found!")
            self.results_df = pd.DataFrame()
            self.total_transaction_costs = 0
            self.total_turnover = 0
            return {}
        
        # Create results DataFrame
        self.results_df = pd.DataFrame(results)
        self.results_df.set_index('date', inplace=True)
        
        # Calculate daily returns
        self.results_df['daily_return'] = self.results_df['portfolio_value'].pct_change()
        
        # Store summary statistics
        self.total_transaction_costs = total_costs
        self.total_turnover = total_turnover
        
        rebalance_count = self.results_df['rebalanced'].sum()
        print(f"\n✅ Backtest complete!")
        print(f"   Final portfolio value: ${portfolio_value:,.2f}")
        print(f"   Total return: {(portfolio_value/self.initial_capital - 1)*100:.2f}%")
        print(f"   Total rebalances: {rebalance_count} (instead of {len(valid_dates)} daily)")
        print(f"   Turnover reduction: {(1 - rebalance_count/len(valid_dates))*100:.1f}%")
        
        return self.calculate_metrics()
    
    def calculate_metrics(self):
        """
        Calculate comprehensive performance metrics.
        """
        
        df = self.results_df.copy()
        
        # Validate we have data
        if len(df) == 0:
            print("⚠️ No results to calculate metrics from!")
            return {}
        
        returns = df['daily_return'].dropna()
        
        if len(returns) == 0:
            print("⚠️ No valid returns to calculate metrics from!")
            return {}
        
        # Basic metrics
        total_return = (df['portfolio_value'].iloc[-1] / self.initial_capital) - 1
        
        # Annualized metrics (assuming 252 trading days)
        n_days = len(returns)
        n_years = max(n_days / 252, 0.001)  # Avoid divide by zero
        
        # Handle potential issues with annualized return calculation
        try:
            if total_return > -1:  # Can only take root if total_return > -1
                annualized_return = (1 + total_return) ** (1/n_years) - 1
            else:
                annualized_return = -1  # Lost everything
        except:
            annualized_return = 0
            
        annualized_volatility = returns.std() * np.sqrt(252) if len(returns) > 1 else 0
        
        # Sharpe Ratio (assuming risk-free rate of 0 for simplicity)
        sharpe_ratio = annualized_return / annualized_volatility if annualized_volatility > 0 else 0
        
        # Sortino Ratio (downside deviation only)
        downside_returns = returns[returns < 0]
        downside_deviation = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 1 else 0
        sortino_ratio = annualized_return / downside_deviation if downside_deviation > 0 else 0
        
        # Maximum Drawdown
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_drawdown = drawdown.min() if len(drawdown) > 0 else 0
        avg_drawdown = drawdown[drawdown < 0].mean() if len(drawdown[drawdown < 0]) > 0 else 0
        
        # Calmar Ratio
        calmar_ratio = annualized_return / abs(max_drawdown) if max_drawdown != 0 else 0
        
        # Win Rate
        win_rate = (returns > 0).mean() if len(returns) > 0 else 0
        
        # Profit Factor
        gross_profits = returns[returns > 0].sum()
        gross_losses = abs(returns[returns < 0].sum())
        profit_factor = gross_profits / gross_losses if gross_losses > 0 else (np.inf if gross_profits > 0 else 1.0)
        
        # Turnover (average daily turnover as % of portfolio)
        avg_turnover = df['turnover'].mean() / df['portfolio_value'].mean() * 100 if df['portfolio_value'].mean() > 0 else 0
        annual_turnover = avg_turnover * 252
        
        # Additional statistics
        skewness = skew(returns)
        excess_kurtosis = kurtosis(returns)
        
        metrics = {
            'Total Return (%)': total_return * 100,
            'Annualized Return (%)': annualized_return * 100,
            'Annualized Volatility (%)': annualized_volatility * 100,
            'Sharpe Ratio': sharpe_ratio,
            'Sortino Ratio': sortino_ratio,
            'Calmar Ratio': calmar_ratio,
            'Maximum Drawdown (%)': max_drawdown * 100,
            'Average Drawdown (%)': avg_drawdown * 100,
            'Win Rate (%)': win_rate * 100,
            'Profit Factor': profit_factor,
            'Daily Turnover (%)': avg_turnover,
            'Annual Turnover (%)': annual_turnover,
            'Total Transaction Costs ($)': self.total_transaction_costs,
            'Skewness': skewness,
            'Excess Kurtosis': excess_kurtosis,
            'Final Portfolio Value ($)': df['portfolio_value'].iloc[-1],
            'Total Profit/Loss ($)': df['portfolio_value'].iloc[-1] - self.initial_capital,
            'Number of Trading Days': n_days
        }
        
        self.metrics = metrics
        return metrics
    
    def plot_performance(self, benchmark_values=None, title="Strategy Performance"):
        """
        Create comprehensive performance visualization.
        """
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        # 1. Cumulative Portfolio Value
        ax1 = axes[0, 0]
        ax1.plot(self.results_df.index, self.results_df['portfolio_value'], 
                 label='Strategy', linewidth=2, color='blue')
        if benchmark_values is not None:
            ax1.plot(benchmark_values.index, benchmark_values.values, 
                     label='Benchmark', linewidth=2, color='gray', alpha=0.7)
        ax1.axhline(y=self.initial_capital, color='red', linestyle='--', 
                    label='Initial Capital', alpha=0.5)
        ax1.set_title('Cumulative Portfolio Value', fontweight='bold', fontsize=12)
        ax1.set_xlabel('Date')
        ax1.set_ylabel('Portfolio Value ($)')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 2. Drawdown
        ax2 = axes[0, 1]
        returns = self.results_df['daily_return'].dropna()
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max * 100
        ax2.fill_between(drawdown.index, drawdown.values, 0, color='red', alpha=0.3)
        ax2.plot(drawdown.index, drawdown.values, color='red', linewidth=1)
        ax2.set_title('Drawdown (%)', fontweight='bold', fontsize=12)
        ax2.set_xlabel('Date')
        ax2.set_ylabel('Drawdown (%)')
        ax2.grid(True, alpha=0.3)
        
        # 3. Daily Returns Distribution
        ax3 = axes[1, 0]
        ax3.hist(returns * 100, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
        ax3.axvline(x=0, color='red', linestyle='--', linewidth=2)
        ax3.axvline(x=returns.mean() * 100, color='green', linestyle='-', linewidth=2, label=f'Mean: {returns.mean()*100:.3f}%')
        ax3.set_title('Daily Returns Distribution', fontweight='bold', fontsize=12)
        ax3.set_xlabel('Daily Return (%)')
        ax3.set_ylabel('Frequency')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # 4. Rolling Sharpe Ratio (60-day)
        ax4 = axes[1, 1]
        rolling_sharpe = (returns.rolling(60).mean() / returns.rolling(60).std()) * np.sqrt(252)
        ax4.plot(rolling_sharpe.index, rolling_sharpe.values, color='purple', linewidth=1)
        ax4.axhline(y=0, color='red', linestyle='--', alpha=0.5)
        ax4.axhline(y=1, color='green', linestyle='--', alpha=0.5, label='Sharpe = 1')
        ax4.axhline(y=2, color='green', linestyle=':', alpha=0.5, label='Sharpe = 2')
        ax4.set_title('Rolling 60-Day Sharpe Ratio', fontweight='bold', fontsize=12)
        ax4.set_xlabel('Date')
        ax4.set_ylabel('Sharpe Ratio')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()
        
        return fig

print("✅ Backtester class defined!")

---

## 📈 Step 4: Run Backtest with Transaction Costs

In [ ]:
# Initialize backtester with 10 bps transaction cost
# Using weekly rebalancing (every 5 days) to reduce turnover
bt_with_costs = Backtester(
    initial_capital=1_000_000,
    transaction_cost=0.001,  # 10 bps = 0.1%
    rebalance_frequency=5    # Only rebalance every 5 trading days (weekly)
)

# Run backtest
metrics_with_costs = bt_with_costs.run(
    signals, prices, ticker_col, date_col, close_col
)

In [ ]:
# Display performance metrics
print("📊 Performance Metrics (WITH Transaction Costs - 10 bps):")
print("="*60)

for metric, value in metrics_with_costs.items():
    if isinstance(value, float):
        print(f"{metric}: {value:,.4f}")
    else:
        print(f"{metric}: {value:,}")

In [ ]:
# Visualize performance
bt_with_costs.plot_performance(title="Strategy Performance (With 10 bps Transaction Costs)")

---

## 📈 Step 5: Run Backtest WITHOUT Transaction Costs (for comparison)

In [ ]:
# Initialize backtester with NO transaction cost
# Using same weekly rebalancing for fair comparison
bt_no_costs = Backtester(
    initial_capital=1_000_000,
    transaction_cost=0.0,     # No transaction costs
    rebalance_frequency=5     # Match rebalancing frequency
)

# Run backtest
metrics_no_costs = bt_no_costs.run(
    signals, prices, ticker_col, date_col, close_col
)

# Display comparison
print("\n📊 Performance Metrics (WITHOUT Transaction Costs):")
print("="*60)

for metric, value in metrics_no_costs.items():
    if isinstance(value, float):
        print(f"{metric}: {value:,.4f}")

    else:        print(f"{metric}: {value:,}")

In [ ]:
# Compare metrics
print("📊 Transaction Cost Impact Analysis:")
print("="*60)

comparison_metrics = ['Total Return (%)', 'Annualized Return (%)', 'Sharpe Ratio', 
                      'Maximum Drawdown (%)', 'Final Portfolio Value ($)', 'Total Transaction Costs ($)']

comparison_df = pd.DataFrame({
    'Without TC': [metrics_no_costs[m] for m in comparison_metrics],
    'With 10 bps TC': [metrics_with_costs[m] for m in comparison_metrics]
}, index=comparison_metrics)

comparison_df['Difference'] = comparison_df['With 10 bps TC'] - comparison_df['Without TC']

display(comparison_df)

---

## 📊 Step 6: Create Equal-Weight Benchmark

In [ ]:
def create_equal_weight_benchmark(prices_df, signals_df, ticker_col, date_col, close_col,
                                   initial_capital=1_000_000, transaction_cost=0.001):
    """
    Create an equal-weight buy-and-hold benchmark.
    Buys all stocks equally at the start and holds throughout.
    """
    
    # Get dates from signals (test period)
    signal_dates = sorted(signals_df[date_col].unique())
    start_date = signal_dates[0]
    
    # Get all unique tickers
    all_tickers = signals_df[ticker_col].unique()
    
    # Filter prices to test period
    test_prices = prices_df[prices_df[date_col] >= start_date].copy()
    
    # Pivot prices
    price_pivot = test_prices.pivot(index=date_col, columns=ticker_col, values=close_col)
    price_pivot = price_pivot[all_tickers].dropna(axis=1, how='all')
    
    # Get first day prices for initial allocation
    first_prices = price_pivot.iloc[0]
    valid_tickers = first_prices.dropna().index.tolist()
    
    # Equal allocation (minus initial transaction costs)
    capital_after_costs = initial_capital * (1 - transaction_cost)
    allocation_per_stock = capital_after_costs / len(valid_tickers)
    
    # Calculate shares for each stock
    shares = {}
    for ticker in valid_tickers:
        if first_prices[ticker] > 0:
            shares[ticker] = allocation_per_stock / first_prices[ticker]
    
    # Calculate daily portfolio value
    benchmark_values = []
    
    for date in price_pivot.index:
        daily_value = 0
        for ticker, n_shares in shares.items():
            if ticker in price_pivot.columns and not pd.isna(price_pivot.loc[date, ticker]):
                daily_value += n_shares * price_pivot.loc[date, ticker]
        benchmark_values.append({'date': date, 'value': daily_value})
    
    benchmark_df = pd.DataFrame(benchmark_values)
    benchmark_df.set_index('date', inplace=True)
    
    return benchmark_df

print("✅ Benchmark function defined!")

In [ ]:
# Create benchmark
benchmark = create_equal_weight_benchmark(
    prices, signals, ticker_col, date_col, close_col,
    initial_capital=1_000_000, transaction_cost=0.001
)

print(f"✅ Benchmark created: {len(benchmark)} days")
print(f"   Start value: ${benchmark['value'].iloc[0]:,.2f}")
print(f"   End value: ${benchmark['value'].iloc[-1]:,.2f}")
print(f"   Total return: {(benchmark['value'].iloc[-1] / benchmark['value'].iloc[0] - 1) * 100:.2f}%")

In [ ]:
# Calculate benchmark metrics
benchmark_returns = benchmark['value'].pct_change().dropna()
benchmark_total_return = (benchmark['value'].iloc[-1] / 1_000_000) - 1
n_years = len(benchmark_returns) / 252
benchmark_ann_return = (1 + benchmark_total_return) ** (1/n_years) - 1 if n_years > 0 else 0
benchmark_ann_vol = benchmark_returns.std() * np.sqrt(252)
benchmark_sharpe = benchmark_ann_return / benchmark_ann_vol if benchmark_ann_vol > 0 else 0

cumulative_bm = (1 + benchmark_returns).cumprod()
running_max_bm = cumulative_bm.expanding().max()
drawdown_bm = (cumulative_bm - running_max_bm) / running_max_bm
max_dd_bm = drawdown_bm.min()

print("📊 Equal-Weight Benchmark Metrics:")
print("="*60)
print(f"Total Return: {benchmark_total_return * 100:.2f}%")
print(f"Annualized Return: {benchmark_ann_return * 100:.2f}%")
print(f"Annualized Volatility: {benchmark_ann_vol * 100:.2f}%")
print(f"Sharpe Ratio: {benchmark_sharpe:.4f}")
print(f"Maximum Drawdown: {max_dd_bm * 100:.2f}%")

---

## 📈 Step 7: Strategy vs Benchmark Comparison

In [ ]:
# Plot Strategy vs Benchmark
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Cumulative Portfolio Value Comparison
ax1 = axes[0, 0]
ax1.plot(bt_with_costs.results_df.index, bt_with_costs.results_df['portfolio_value'], 
         label='Strategy (with TC)', linewidth=2, color='blue')
ax1.plot(bt_no_costs.results_df.index, bt_no_costs.results_df['portfolio_value'], 
         label='Strategy (no TC)', linewidth=2, color='lightblue', linestyle='--')
ax1.plot(benchmark.index, benchmark['value'], 
         label='Equal-Weight Benchmark', linewidth=2, color='gray')
ax1.axhline(y=1_000_000, color='red', linestyle=':', alpha=0.5, label='Initial Capital')
ax1.set_title('Cumulative Portfolio Value: Strategy vs Benchmark', fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('Portfolio Value ($)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Cumulative Returns (Normalized) - with ROBUST alignment
ax2 = axes[0, 1]
strategy_cum_return = (bt_with_costs.results_df['portfolio_value'] / 1_000_000 - 1) * 100
benchmark_cum_return = (benchmark['value'] / 1_000_000 - 1) * 100

# Align to common dates
common_idx = strategy_cum_return.index.intersection(benchmark_cum_return.index)
if len(common_idx) > 0:
    strategy_cum_aligned = strategy_cum_return.loc[common_idx].values
    benchmark_cum_aligned = benchmark_cum_return.loc[common_idx].values
    common_dates_plot = common_idx
    
    ax2.plot(common_dates_plot, strategy_cum_aligned, label='Strategy', linewidth=2, color='blue')
    ax2.plot(common_dates_plot, benchmark_cum_aligned, label='Benchmark', linewidth=2, color='gray')
    ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    
    # Safe fill_between with explicit numpy arrays
    try:
        ax2.fill_between(common_dates_plot, strategy_cum_aligned, benchmark_cum_aligned,
                          where=strategy_cum_aligned > benchmark_cum_aligned, 
                          color='green', alpha=0.3, label='Outperformance')
        ax2.fill_between(common_dates_plot, strategy_cum_aligned, benchmark_cum_aligned,
                          where=strategy_cum_aligned <= benchmark_cum_aligned, 
                          color='red', alpha=0.3, label='Underperformance')
    except Exception as e:
        print(f"⚠️ Could not plot fill_between: {e}")
else:
    # No common dates - plot separately
    ax2.plot(strategy_cum_return.index, strategy_cum_return.values, label='Strategy', linewidth=2, color='blue')
    ax2.plot(benchmark_cum_return.index, benchmark_cum_return.values, label='Benchmark', linewidth=2, color='gray')
    ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5)

ax2.set_title('Cumulative Returns (%)', fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Cumulative Return (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Rolling Outperformance (Alpha)
ax3 = axes[1, 0]
strategy_returns = bt_with_costs.results_df['daily_return'].dropna()

# Align dates with error handling
common_dates = strategy_returns.index.intersection(benchmark_returns.index)
if len(common_dates) > 0:
    strategy_aligned = strategy_returns.loc[common_dates]
    benchmark_aligned = benchmark_returns.loc[common_dates]
    
    excess_returns = strategy_aligned - benchmark_aligned
    rolling_alpha = excess_returns.rolling(min(60, len(excess_returns))).mean() * 252 * 100
    
    ax3.plot(rolling_alpha.index, rolling_alpha.values, color='purple', linewidth=1)
    ax3.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    try:
        ax3.fill_between(rolling_alpha.index, rolling_alpha.values, 0,
                          where=rolling_alpha.values > 0, color='green', alpha=0.3)
        ax3.fill_between(rolling_alpha.index, rolling_alpha.values, 0,
                          where=rolling_alpha.values <= 0, color='red', alpha=0.3)
    except:
        pass
else:
    ax3.text(0.5, 0.5, 'No overlapping dates', ha='center', va='center', transform=ax3.transAxes)

ax3.set_title('Rolling 60-Day Annualized Alpha (%)', fontweight='bold')
ax3.set_xlabel('Date')
ax3.set_ylabel('Alpha (%)')
ax3.grid(True, alpha=0.3)

# 4. Metrics Comparison Bar Chart
ax4 = axes[1, 1]
metrics_to_compare = ['Sharpe Ratio', 'Total Return (%)', 'Max Drawdown (%)']

# Handle potential NaN values in metrics
def safe_get(d, key, default=0):
    val = d.get(key, default)
    return default if pd.isna(val) else val

strategy_vals = [safe_get(metrics_with_costs, 'Sharpe Ratio', 0), 
                 safe_get(metrics_with_costs, 'Total Return (%)', 0),
                 abs(safe_get(metrics_with_costs, 'Maximum Drawdown (%)', 0))]
benchmark_vals = [benchmark_sharpe if not pd.isna(benchmark_sharpe) else 0, 
                  benchmark_total_return * 100 if not pd.isna(benchmark_total_return) else 0,
                  abs(max_dd_bm * 100) if not pd.isna(max_dd_bm) else 0]

x = np.arange(len(metrics_to_compare))
width = 0.35

bars1 = ax4.bar(x - width/2, strategy_vals, width, label='Strategy', color='steelblue')
bars2 = ax4.bar(x + width/2, benchmark_vals, width, label='Benchmark', color='gray')

ax4.set_ylabel('Value')
ax4.set_title('Key Metrics Comparison', fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(metrics_to_compare)
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars1:
    height = bar.get_height()
    if not pd.isna(height):
        ax4.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                     ha='center', va='bottom', fontsize=10)
for bar in bars2:
    height = bar.get_height()
    if not pd.isna(height):
        ax4.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                     ha='center', va='bottom', fontsize=10)

plt.suptitle('Strategy vs Equal-Weight Benchmark', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 📊 Step 8: Monthly Returns Heatmap

In [ ]:
# Create monthly returns heatmap
strategy_returns = bt_with_costs.results_df['daily_return'].dropna()

# Only plot monthly heatmap if we have data
if len(strategy_returns) > 0:
    # Resample to monthly returns
    monthly_returns = (1 + strategy_returns).resample('M').prod() - 1
    monthly_returns = monthly_returns * 100  # Convert to percentage
    
    # Create pivot table for heatmap
    monthly_df = pd.DataFrame({
        'Year': monthly_returns.index.year,
        'Month': monthly_returns.index.month,
        'Return': monthly_returns.values
    })
    
    monthly_pivot = monthly_df.pivot(index='Year', columns='Month', values='Return')
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    monthly_pivot.columns = [month_names[m-1] for m in monthly_pivot.columns]
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(monthly_pivot, annot=True, fmt='.1f', center=0, 
                cmap='RdYlGn', linewidths=0.5, ax=ax,
                cbar_kws={'label': 'Monthly Return (%)'})
    ax.set_title('Monthly Returns Heatmap (%)', fontweight='bold', fontsize=14)
    ax.set_xlabel('Month')
    ax.set_ylabel('Year')
    plt.tight_layout()
    plt.show()
    
    # Summary statistics by month
    print("📊 Average Monthly Returns:")
    print(monthly_pivot.mean())
else:
    print("⚠️ No strategy returns to display in heatmap")

---

## 📝 Step 9: Strategy Analysis and Conclusions

In [ ]:
# Comprehensive Summary - with NaN handling
def safe_fmt(val, fmt=".2f"):
    """Safely format a value, returning 'N/A' for NaN"""
    if pd.isna(val):
        return "N/A"
    return f"{val:{fmt}}"

print("="*80)
print("📊 STRATEGY PERFORMANCE SUMMARY")
print("="*80)

print("\n💰 RETURNS:")
print(f"   Strategy Total Return: {safe_fmt(metrics_with_costs.get('Total Return (%)', 0))}%")
print(f"   Benchmark Total Return: {safe_fmt(benchmark_total_return * 100)}%")
strat_return = metrics_with_costs.get('Total Return (%)', 0) or 0
print(f"   Excess Return (Alpha): {safe_fmt(strat_return - benchmark_total_return * 100)}%")

print("\n📈 RISK-ADJUSTED METRICS:")
print(f"   Strategy Sharpe Ratio: {safe_fmt(metrics_with_costs.get('Sharpe Ratio', 0), '.4f')}")
print(f"   Benchmark Sharpe Ratio: {safe_fmt(benchmark_sharpe, '.4f')}")
print(f"   Strategy Sortino Ratio: {safe_fmt(metrics_with_costs.get('Sortino Ratio', 0), '.4f')}")

print("\n📉 RISK METRICS:")
print(f"   Strategy Max Drawdown: {safe_fmt(metrics_with_costs.get('Maximum Drawdown (%)', 0))}%")
print(f"   Benchmark Max Drawdown: {safe_fmt(max_dd_bm * 100)}%")
print(f"   Strategy Volatility: {safe_fmt(metrics_with_costs.get('Annualized Volatility (%)', 0))}%")

print("\n💸 TRANSACTION COSTS IMPACT:")
print(f"   Total Transaction Costs: ${safe_fmt(metrics_with_costs.get('Total Transaction Costs ($)', 0), ',.2f')}")
no_tc_return = metrics_no_costs.get('Total Return (%)', 0) or 0
with_tc_return = metrics_with_costs.get('Total Return (%)', 0) or 0
print(f"   Return Before TC: {safe_fmt(no_tc_return)}%")
print(f"   Return After TC: {safe_fmt(with_tc_return)}%")
print(f"   TC Impact: {safe_fmt(no_tc_return - with_tc_return)}%")

print("\n🔄 PORTFOLIO TURNOVER:")
print(f"   Average Daily Turnover: {safe_fmt(metrics_with_costs.get('Daily Turnover (%)', 0))}%")
print(f"   Annual Turnover: {safe_fmt(metrics_with_costs.get('Annual Turnover (%)', 0))}%")

print("\n🏆 FINAL VERDICT:")
strat_ret = metrics_with_costs.get('Total Return (%)', 0) or 0
bench_ret = (benchmark_total_return * 100) if not pd.isna(benchmark_total_return) else 0

if strat_ret > bench_ret:
    print("   ✅ Strategy OUTPERFORMED the benchmark!")
else:
    print("   ❌ Strategy UNDERPERFORMED the benchmark.")

sharpe = metrics_with_costs.get('Sharpe Ratio', 0) or 0
if sharpe > 1:
    print("   ✅ Sharpe Ratio > 1 indicates good risk-adjusted returns")
elif sharpe > 0.5:
    print("   ⚠️ Sharpe Ratio between 0.5-1 is moderate")
else:
    print("   ❌ Sharpe Ratio < 0.5 indicates poor risk-adjusted returns")

if strat_ret > 0:
    print("   ✅ Strategy survives transaction costs!")
else:
    print("   ❌ Strategy does NOT survive transaction costs.")

In [ ]:
# Analyze when and why the strategy failed/succeeded
print("\n📊 PERIOD ANALYSIS:")
print("="*60)

strategy_returns = bt_with_costs.results_df['daily_return'].dropna()

if len(strategy_returns) > 0:
    # Calculate monthly returns for analysis
    monthly_returns_analysis = (1 + strategy_returns).resample('M').prod() - 1
    monthly_returns_analysis = monthly_returns_analysis * 100
    
    # Best and worst months
    print("\n🏆 Best Months:")
    best_months = monthly_returns_analysis.nlargest(5)
    for date, ret in best_months.items():
        print(f"   {date.strftime('%Y-%m')}: +{ret:.2f}%")
    
    print("\n💔 Worst Months:")
    worst_months = monthly_returns_analysis.nsmallest(5)
    for date, ret in worst_months.items():
        print(f"   {date.strftime('%Y-%m')}: {ret:.2f}%")
    
    # Drawdown periods
    print("\n📉 Major Drawdown Periods:")
    cumulative = (1 + strategy_returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    
    # Find drawdown periods
    major_dds = drawdown[drawdown < -0.05].sort_values()
    if len(major_dds) > 0:
        print(f"   Deepest drawdown: {major_dds.iloc[0]*100:.2f}% on {major_dds.index[0].strftime('%Y-%m-%d')}")
    else:
        print("   No major drawdowns (>5%) detected!")
else:
    print("⚠️ No strategy returns data available for period analysis")

---

## 💾 Step 10: Save Results

In [ ]:
# Save backtest results
if len(bt_with_costs.results_df) > 0:
    bt_with_costs.results_df.to_csv('backtest_results.csv')
    print("✅ Saved backtest_results.csv")
else:
    print("⚠️ No backtest results to save")

benchmark.to_csv('benchmark_results.csv')

# Save metrics to JSON - handle NaN values
import json

def clean_for_json(obj):
    """Convert NaN and inf to None for JSON serialization"""
    if isinstance(obj, dict):
        return {k: clean_for_json(v) for k, v in obj.items()}
    elif isinstance(obj, float):
        if pd.isna(obj) or np.isinf(obj):
            return None
        return obj
    return obj

all_metrics = {
    'strategy_with_tc': clean_for_json(metrics_with_costs),
    'strategy_no_tc': clean_for_json(metrics_no_costs),
    'benchmark': clean_for_json({
        'Total Return (%)': benchmark_total_return * 100,
        'Annualized Return (%)': benchmark_ann_return * 100,
        'Annualized Volatility (%)': benchmark_ann_vol * 100,
        'Sharpe Ratio': benchmark_sharpe,
        'Maximum Drawdown (%)': max_dd_bm * 100
    })
}

with open('performance_metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=4, default=str)

print("✅ Results saved!")
print("   - backtest_results.csv")
print("   - benchmark_results.csv")
print("   - performance_metrics.json")

---

## 📝 Summary & Key Findings

### What We Accomplished:

1. **Built a Comprehensive Backtester:**
   - Long-short portfolio simulation
   - Realistic transaction cost modeling
   - Daily position tracking

2. **Calculated Key Metrics:**
   - Sharpe Ratio (annualized)
   - Maximum Drawdown & Average Drawdown
   - Portfolio Turnover
   - Total Return & Profit/Loss

3. **Benchmark Comparison:**
   - Created equal-weight buy-and-hold benchmark
   - Analyzed alpha generation
   - Visualized outperformance periods

4. **Transaction Cost Analysis:**
   - Compared performance with/without costs
   - Quantified cost impact on returns

### Key Insights:

- The strategy's ability to survive transaction costs depends on alpha quality
- High turnover strategies are more sensitive to trading costs
- Long-short strategies can provide downside protection in market corrections
- Consistent positive alpha is more valuable than extreme returns

### Next Steps (Part 4):
- Identify correlated/cointegrated asset pairs
- Build statistical arbitrage signals
- Propose pairs trading overlay

---